# UdaPlay - Part 2: Agent Implementation

In this notebook, we build the full UdaPlay AI Research Agent that:

1. Answers user questions using a two-tier retrieval system (RAG + Web Search)
2. Implements three core tools: `retrieve_game`, `evaluate_retrieval`, `game_web_search`
3. Uses a state machine for agent workflow management
4. Maintains conversation state across multiple queries
5. Generates clean, structured, and well-cited answers

## 1. Environment Setup

In [1]:
# Load environment variables
from dotenv import load_dotenv
import os

load_dotenv("config.env")

assert os.getenv("OPENAI_API_KEY") is not None, "OPENAI_API_KEY not found"
assert os.getenv("TAVILY_API_KEY") is not None, "TAVILY_API_KEY not found"

OPENAI_BASE_URL = os.getenv(
    "OPENAI_BASE_URL",
    "https://openai.vocareum.com/v1"
)

print("Environment variables loaded successfully!")

Environment variables loaded successfully!


In [2]:
# Import required libraries
import json
import hashlib
from enum import Enum
from typing import List, Dict, Optional, Any
from datetime import datetime

import chromadb
from openai import OpenAI
from pydantic import BaseModel, Field
from tavily import TavilyClient

print("Libraries imported.")

Libraries imported.


## 2. Reconnect to Vector Store

We reconnect to the ChromaDB vector store created in Part 1.

In [3]:
class VectorStoreManager:
    """Manages the ChromaDB vector store for game information."""
    
    def __init__(self, persist_directory: str = "./chroma_db", collection_name: str = "udaplay_games"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        
        self.openai_client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=OPENAI_BASE_URL
        )
        
        self.chroma_client = chromadb.PersistentClient(path=persist_directory)
        self.collection = self.chroma_client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        
        print(f"Connected to vector store. Collection '{collection_name}' has {self.collection.count()} documents.")
    
    def get_embeddings(self, texts: List[str]) -> List[List[float]]:
        response = self.openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=texts
        )
        return [item.embedding for item in response.data]
    
    def add_documents(self, documents: List[str], metadatas: List[Dict], ids: List[str]):
        embeddings = self.get_embeddings(documents)
        self.collection.upsert(
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids
        )
    
    def search(self, query: str, n_results: int = 3, filter_type: Optional[str] = None) -> Dict:
        query_embedding = self.get_embeddings([query])[0]
        
        query_params = {
            "query_embeddings": [query_embedding],
            "n_results": n_results,
            "include": ["documents", "metadatas", "distances"]
        }
        
        if filter_type:
            query_params["where"] = {"type": filter_type}
        
        results = self.collection.query(**query_params)
        
        return {
            "documents": results["documents"][0] if results["documents"] else [],
            "metadatas": results["metadatas"][0] if results["metadatas"] else [],
            "distances": results["distances"][0] if results["distances"] else []
        }

# Initialize the vector store (will use existing data from Part 1)
vector_store = VectorStoreManager(
    persist_directory="./chroma_db",
    collection_name="udaplay_games"
)

Connected to vector store. Collection 'udaplay_games' has 25 documents.


In [4]:
# If the vector store is empty (Part 1 not run yet), populate it
if vector_store.collection.count() == 0:
    print("Vector store is empty. Loading data...")
    
    # Load game data
    with open("data/games/games.json", 'r') as f:
        games_data = json.load(f)
    with open("data/games/companies.json", 'r') as f:
        companies_data = json.load(f)
    
    documents = []
    metadatas = []
    ids = []
    
    for game in games_data:
        platforms_str = ", ".join(game["platforms"])
        doc = (
            f"Game Title: {game['title']}\n"
            f"Developer: {game['developer']}\n"
            f"Publisher: {game['publisher']}\n"
            f"Release Date: {game['release_date']}\n"
            f"Platforms: {platforms_str}\n"
            f"Genre: {game['genre']}\n"
            f"Description: {game['description']}"
        )
        documents.append(doc)
        metadatas.append({
            "type": "game",
            "title": game["title"],
            "developer": game["developer"],
            "publisher": game["publisher"],
            "release_date": game["release_date"],
            "genre": game["genre"]
        })
        ids.append(f"game_{hashlib.md5(game['title'].encode()).hexdigest()}")
    
    for company in companies_data:
        franchises_str = ", ".join(company["notable_franchises"])
        doc = (
            f"Company Name: {company['name']}\n"
            f"Headquarters: {company['headquarters']}\n"
            f"Founded: {company['founded']}\n"
            f"Notable Franchises: {franchises_str}\n"
            f"Current Projects: {company['current_projects']}\n"
            f"Description: {company['description']}"
        )
        documents.append(doc)
        metadatas.append({
            "type": "company",
            "name": company["name"],
            "headquarters": company["headquarters"],
            "founded": company["founded"]
        })
        ids.append(f"company_{hashlib.md5(company['name'].encode()).hexdigest()}")
    
    vector_store.add_documents(documents, metadatas, ids)
    print(f"Loaded {len(documents)} documents into vector store.")
else:
    print(f"Vector store already contains {vector_store.collection.count()} documents. Ready to use.")

Vector store already contains 25 documents. Ready to use.


## 3. Define Agent State Machine

The agent uses a state machine to manage its workflow:

0. **REWRITE** - Resolve pronouns/references in follow-up queries using conversation history
1. **RETRIEVE** - Search internal knowledge base
2. **EVALUATE** - Assess quality of retrieved information
3. **WEB_SEARCH** - Fall back to web search if needed
4. **GENERATE** - Create final structured answer
5. **COMPLETE** - Return the answer

In [5]:
class AgentState(Enum):
    """States in the agent's workflow state machine."""
    RETRIEVE = "retrieve"
    EVALUATE = "evaluate"
    WEB_SEARCH = "web_search"
    GENERATE = "generate"
    COMPLETE = "complete"

class RetrievalResult(BaseModel):
    """Model for retrieval results."""
    documents: List[str] = Field(default_factory=list)
    metadatas: List[Dict] = Field(default_factory=list)
    distances: List[float] = Field(default_factory=list)
    source: str = "internal_db"

class EvaluationResult(BaseModel):
    """Model for evaluation results."""
    is_sufficient: bool
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str
    needs_web_search: bool

class AgentResponse(BaseModel):
    """Model for the final agent response."""
    answer: str
    sources: List[str] = Field(default_factory=list)
    confidence: float = Field(ge=0.0, le=1.0)
    tools_used: List[str] = Field(default_factory=list)
    reasoning_steps: List[str] = Field(default_factory=list)

print("Agent state models ready.")

Agent state models ready.


## 4. Implement Agent Tools

The agent has three core tools:
1. **retrieve_game** - Search the vector database for game/company info
2. **evaluate_retrieval** - Assess quality of retrieved results
3. **game_web_search** - Search the web using Tavily API

In [6]:
class AgentTools:
    """Collection of tools available to the UdaPlay agent."""
    
    def __init__(self, vector_store: VectorStoreManager):
        self.vector_store = vector_store
        self.openai_client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=OPENAI_BASE_URL
        )
        self.tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    
    def retrieve_game(self, query: str, n_results: int = 3) -> RetrievalResult:
        """
        Tool 1: Retrieve game information from the internal vector database.
        
        Searches the ChromaDB vector store for relevant game or company information
        based on semantic similarity to the query.
        
        Args:
            query: Natural language query about a game or company
            n_results: Number of results to retrieve
            
        Returns:
            RetrievalResult with documents, metadata, and similarity scores
        """
        print(f"  [retrieve_game] query='{query[:80]}' | ", end="")
        
        results = self.vector_store.search(query, n_results=n_results)
        
        retrieval = RetrievalResult(
            documents=results["documents"],
            metadatas=results["metadatas"],
            distances=results["distances"],
            source="internal_db"
        )
        
        best_sim = (1 - retrieval.distances[0]) if retrieval.distances else 0
        print(f"{len(retrieval.documents)} results, best_similarity={best_sim:.4f}")
        
        return retrieval
    
    def evaluate_retrieval(self, query: str, retrieval: RetrievalResult) -> EvaluationResult:
        """
        Tool 2: Evaluate the quality of retrieved results.
        
        Uses an LLM to assess whether the retrieved information is sufficient
        to answer the user's query with high confidence.
        
        Args:
            query: The original user query
            retrieval: The retrieval results to evaluate
            
        Returns:
            EvaluationResult with confidence score and recommendation
        """
        print(f"  [evaluate_retrieval] ", end="")
        
        # Check if we have any results
        if not retrieval.documents:
            return EvaluationResult(
                is_sufficient=False,
                confidence=0.0,
                reasoning="No documents retrieved from the database.",
                needs_web_search=True
            )
        
        # Use the LLM to evaluate relevance
        context = "\n\n---\n\n".join(retrieval.documents[:3])
        
        evaluation_prompt = f"""You are an evaluation system. Assess whether the following retrieved context 
contains sufficient information to answer the user's question accurately and completely.

User Question: {query}

Retrieved Context:
{context}

Evaluate and respond in this exact JSON format:
{{
    "is_sufficient": true/false,
    "confidence": 0.0 to 1.0,
    "reasoning": "explanation of your assessment",
    "needs_web_search": true/false
}}

Rules:
- confidence >= 0.7 means the context is sufficient
- confidence < 0.7 means web search is recommended
- Consider whether the specific information asked for is explicitly present in the context
- If the question asks about something not covered in the context, set needs_web_search to true"""

        response = self.openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a precise evaluation system. Always respond with valid JSON."},
                {"role": "user", "content": evaluation_prompt}
            ],
            temperature=0.1
        )
        
        try:
            eval_text = response.choices[0].message.content.strip()
            # Handle potential markdown code block wrapping
            if eval_text.startswith("```"):
                eval_text = eval_text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
            eval_data = json.loads(eval_text)
            result = EvaluationResult(**eval_data)
        except (json.JSONDecodeError, Exception) as e:
            # Fallback: use distance-based heuristic
            best_similarity = 1 - retrieval.distances[0] if retrieval.distances else 0
            result = EvaluationResult(
                is_sufficient=best_similarity >= 0.7,
                confidence=best_similarity,
                reasoning=f"Fallback evaluation based on similarity score: {best_similarity:.4f}",
                needs_web_search=best_similarity < 0.7
            )
        
        print(f"confidence={result.confidence:.2f}, sufficient={result.is_sufficient}, web_search={result.needs_web_search}")
        
        return result
    
    def game_web_search(self, query: str, max_results: int = 5) -> Dict[str, Any]:
        """
        Tool 3: Search the web for game information using Tavily API.
        
        Falls back to web search when internal knowledge is insufficient.
        
        Args:
            query: Search query string
            max_results: Maximum number of results to return
            
        Returns:
            Dictionary with search results including titles, URLs, and content
        """
        print(f"  [game_web_search] query='{query[:60]}' | ", end="")
        
        try:
            response = self.tavily_client.search(
                query=query,
                max_results=max_results,
                search_depth="basic",
                include_answer=True
            )
            
            results = {
                "answer": response.get("answer", ""),
                "sources": [],
                "raw_content": []
            }
            
            for result in response.get("results", []):
                results["sources"].append({
                    "title": result.get("title", ""),
                    "url": result.get("url", ""),
                    "content": result.get("content", "")
                })
                results["raw_content"].append(result.get("content", ""))
            
            print(f"{len(results['sources'])} results")
            return results
            
        except Exception as e:
            print(f"ERROR: {str(e)}")
            return {
                "answer": "",
                "sources": [],
                "raw_content": [],
                "error": str(e)
            }

print("AgentTools ready: retrieve_game, evaluate_retrieval, game_web_search")

AgentTools ready: retrieve_game, evaluate_retrieval, game_web_search


## 5. Build the UdaPlay Agent

The agent class implements a state machine that orchestrates the tools and manages conversation history.

In [7]:
class UdaPlayAgent:
    """
    UdaPlay AI Research Agent for video game information.
    
    Implements a stateful agent with a state machine workflow:
    REWRITE -> RETRIEVE -> EVALUATE -> (WEB_SEARCH if needed) -> GENERATE -> COMPLETE
    
    Features:
    - Two-tier information retrieval (RAG + Web Search)
    - Quality evaluation of retrieved information
    - Persistent conversation memory
    - Structured, cited responses
    """
    
    def __init__(self, vector_store: VectorStoreManager):
        """
        Initialize the UdaPlay agent.
        
        Args:
            vector_store: VectorStoreManager instance for RAG
        """
        self.tools = AgentTools(vector_store)
        self.vector_store = vector_store
        self.openai_client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=OPENAI_BASE_URL
        )
        
        # Conversation history for stateful interaction
        self.conversation_history: List[Dict[str, str]] = []
        
        # Long-term memory: stores information learned from web searches
        self.memory: List[Dict[str, Any]] = []
        
        # Agent state tracking
        self.current_state = AgentState.RETRIEVE
        self.state_history: List[str] = []
        
        print("UdaPlay Agent initialized and ready!")
    
    def _transition_state(self, new_state: AgentState):
        """Transition the agent to a new state."""
        self.state_history.append(f"{self.current_state.value} -> {new_state.value}")
        self.current_state = new_state
    
    def _rewrite_query(self, query: str) -> str:
        """
        Rewrite a query to resolve pronouns and references using conversation history.
        
        If the query contains references like 'that game', 'the last one', etc.,
        this method uses the LLM to produce a self-contained query that can be
        used for retrieval without conversation context.
        
        Args:
            query: The raw user query (may contain unresolved references)
            
        Returns:
            A rewritten, self-contained query suitable for vector search
        """
        if not self.conversation_history:
            return query
        
        # Build conversation context: include all user queries for ordinal references
        # and recent exchanges for contextual references
        user_queries = [msg['content'][:150] for msg in self.conversation_history if msg['role'] == 'user']
        queries_summary = "\n".join(f"  Q{i+1}: {q}" for i, q in enumerate(user_queries))
        
        recent = self.conversation_history[-6:]  # Last 3 exchanges for detailed context
        history_str = "\n".join(
            f"{msg['role'].capitalize()}: {msg['content'][:200]}" for msg in recent
        )
        
        rewrite_prompt = f"""Given the conversation history below, rewrite the user's latest query 
into a fully self-contained search query that resolves all pronouns and references 
(e.g., 'that game', 'the last one', 'the first game', 'its developer') into explicit names.

All user queries in order:
{queries_summary}

Recent conversation context:
{history_str}

Latest query: {query}

Rewritten query (return ONLY the rewritten query, nothing else):"""
        
        response = self.openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You rewrite queries to be self-contained. Output only the rewritten query."},
                {"role": "user", "content": rewrite_prompt}
            ],
            temperature=0.0
        )
        
        rewritten = response.choices[0].message.content.strip()
        if rewritten and rewritten != query:
            print(f"  [Query Rewrite] '{query}' -> '{rewritten}'")
            return rewritten
        return query
    
    def _persist_to_memory(self, query: str, answer: str, source: str):
        """
        Persist information to long-term memory.
        Useful when web search provides new information not in the vector store.
        """
        memory_entry = {
            "query": query,
            "answer": answer,
            "source": source,
            "timestamp": datetime.now().isoformat()
        }
        self.memory.append(memory_entry)
        
        # Also add to vector store for future retrieval
        doc = f"Query: {query}\nAnswer: {answer}\nSource: {source}"
        doc_id = f"memory_{hashlib.md5(query.encode()).hexdigest()}"
        self.vector_store.add_documents(
            documents=[doc],
            metadatas=[{"type": "memory", "query": query, "source": source}],
            ids=[doc_id]
        )
    
    def _generate_answer(self, query: str, context: str, sources: List[str], resolved_query: Optional[str] = None) -> str:
        """
        Generate a structured answer using the LLM.
        
        Args:
            query: The user's original question
            context: Combined context from retrieval and/or web search
            sources: List of source descriptions
            resolved_query: The rewritten query with resolved references (if different from query)
            
        Returns:
            Generated answer string
        """
        # Include conversation history for context
        history_context = ""
        if self.conversation_history:
            recent_history = self.conversation_history[-6:]  # Last 3 exchanges
            history_context = "\nPrevious conversation:\n"
            for msg in recent_history:
                history_context += f"{msg['role'].capitalize()}: {msg['content']}\n"
        
        sources_str = "\n".join(f"- {s}" for s in sources)
        
        # If the query was rewritten, include the resolved form to guide generation
        query_section = f"User Question: {query}"
        if resolved_query and resolved_query != query:
            query_section += f"\nInterpreted as: {resolved_query}"
        
        current_date = datetime.now().strftime("%B %d, %Y")
        system_prompt = f"""You are UdaPlay, an expert gaming research assistant.
Today's date is {current_date}.
Generate clear, well-structured answers about video games.

Rules:
1. Always cite your sources at the end of the answer
2. Be specific and factual - answer ONLY based on the provided context
3. If information comes from multiple sources, synthesize it clearly
4. Format the answer in a readable way
5. If you're not fully confident, indicate the level of certainty
6. When referencing current events, use today's date as your frame of reference
7. If the question was rewritten/interpreted, answer the interpreted version"""
        
        user_prompt = f"""{history_context}
Information Sources:
{sources_str}

Context:
{context}

{query_section}

Please provide a comprehensive, well-cited answer based on the context above."""
        
        response = self.openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.3
        )
        
        return response.choices[0].message.content
    
    def process_query(self, query: str) -> AgentResponse:
        """
        Process a user query through the agent's state machine.
        
        Workflow:
        1. RETRIEVE: Search internal vector database
        2. EVALUATE: Assess quality of retrieval
        3. WEB_SEARCH (conditional): Search web if retrieval insufficient
        4. GENERATE: Create structured answer
        5. COMPLETE: Return final response
        
        Args:
            query: The user's natural language question
            
        Returns:
            AgentResponse with answer, sources, confidence, and reasoning
        """
        
        # Reset state for new query
        self.current_state = AgentState.RETRIEVE
        self.state_history = []
        
        tools_used = []
        reasoning_steps = []
        sources = []
        all_context = ""
        confidence = 0.0
        
        # === QUERY REWRITING (resolve pronouns/references) ===
        search_query = self._rewrite_query(query)
        if search_query != query:
            reasoning_steps.append(f"Query rewritten: '{query}' -> '{search_query}'")
        
        # === STATE 1: RETRIEVE ===
        reasoning_steps.append("Searching internal game database...")
        retrieval = self.tools.retrieve_game(search_query)
        tools_used.append("retrieve_game")
        
        if retrieval.documents:
            all_context = "\n\n---\n\n".join(retrieval.documents)
            sources.append("Internal Game Database (ChromaDB)")
        
        # === STATE 2: EVALUATE ===
        self._transition_state(AgentState.EVALUATE)
        evaluation = self.tools.evaluate_retrieval(search_query, retrieval)
        tools_used.append("evaluate_retrieval")
        confidence = evaluation.confidence
        reasoning_steps.append(f"Evaluation: confidence={evaluation.confidence:.2f}, sufficient={evaluation.is_sufficient}")
        
        # === STATE 3: WEB_SEARCH (conditional) ===
        if evaluation.needs_web_search:
            self._transition_state(AgentState.WEB_SEARCH)
            reasoning_steps.append("Internal knowledge insufficient -> web search")
            
            web_results = self.tools.game_web_search(search_query)
            tools_used.append("game_web_search")
            
            if web_results.get("answer"):
                all_context += f"\n\n---\nWeb Search Results:\n{web_results['answer']}"
                sources.append("Web Search (Tavily)")
                
            if web_results.get("raw_content"):
                web_context = "\n".join(web_results["raw_content"][:3])
                all_context += f"\n\nAdditional Web Content:\n{web_context}"
            
            for src in web_results.get("sources", [])[:3]:
                sources.append(f"Web: {src.get('title', 'Unknown')} ({src.get('url', '')})")
            
            if web_results.get("answer") or web_results.get("raw_content"):
                confidence = max(confidence, 0.8)
            
            if web_results.get("answer"):
                self._persist_to_memory(query, web_results["answer"], "web_search")
                reasoning_steps.append("Persisted web results to long-term memory")
        else:
            reasoning_steps.append("Internal knowledge sufficient. Skipping web search.")
        
        # === STATE 4: GENERATE ===
        self._transition_state(AgentState.GENERATE)
        answer = self._generate_answer(query, all_context, sources, search_query)
        
        # === STATE 5: COMPLETE ===
        self._transition_state(AgentState.COMPLETE)
        
        # Update conversation history
        self.conversation_history.append({"role": "user", "content": query})
        self.conversation_history.append({"role": "assistant", "content": answer})
        
        # Build final response
        response = AgentResponse(
            answer=answer,
            sources=sources,
            confidence=confidence,
            tools_used=tools_used,
            reasoning_steps=reasoning_steps
        )
        
        return response
    
    def generate_report(self, response: AgentResponse) -> str:
        """
        Generate a clean, formatted report from an agent response.
        
        Args:
            response: The AgentResponse to format
            
        Returns:
            Formatted report string
        """
        report = []
        report.append("\n" + "=" * 60)
        report.append("UDAPLAY AGENT REPORT")
        report.append("=" * 60)
        
        report.append(f"\n--- Answer ---")
        report.append(response.answer)
        
        report.append(f"\n--- Confidence Level ---")
        confidence_bar = "#" * int(response.confidence * 20) + "-" * (20 - int(response.confidence * 20))
        report.append(f"[{confidence_bar}] {response.confidence:.0%}")
        
        report.append(f"\n--- Sources ---")
        for i, source in enumerate(response.sources, 1):
            report.append(f"  {i}. {source}")
        
        report.append(f"\n--- Tools Used ---")
        report.append(f"  {' -> '.join(response.tools_used)}")
        
        report.append(f"\n--- Reasoning Steps ---")
        for step in response.reasoning_steps:
            report.append(f"  {step}")
        
        report.append("\n" + "=" * 60)
        
        return "\n".join(report)
    
    def ask(self, query: str) -> str:
        """
        High-level method to ask the agent a question and get a formatted report.
        
        Args:
            query: Natural language question
            
        Returns:
            Formatted report string
        """
        response = self.process_query(query)
        report = self.generate_report(response)
        return report

print("UdaPlayAgent ready.")

UdaPlayAgent ready.


## 6. Initialize and Test the Agent

In [8]:
# Initialize the agent
agent = UdaPlayAgent(vector_store)

UdaPlay Agent initialized and ready!


## 7. Run Example Queries

We run 7 queries that exercise different agent capabilities:

| # | Query | Tests |
|---|-------|-------|
| Q1 | Who developed FIFA 21? | Direct retrieval from internal DB |
| Q2 | When was God of War Ragnarok released? | Multi-field retrieval (date + platforms) |
| Q3 | What is Rockstar Games working on right now? | Company info + temporal awareness |
| Q4 | When was Hollow Knight: Silksong released? | Web search fallback (not in DB) |
| Q5 | What other games has that company released? | Pronoun resolution via query rewrite |
| Q6 | What was the first game I asked about? | Ordinal memory across full session |
| Q7 | Recommend a game similar to Hollow Knight | Recommendation + platform constraint |

In [9]:
queries = [
    "Who developed FIFA 21?",                                                          # Q1: Direct retrieval
    "When was God of War Ragnarok released and on which platforms?",                    # Q2: Multi-field
    "What is Rockstar Games working on right now?",                                    # Q3: Temporal awareness
    "When was Hollow Knight: Silksong released and what platforms is it on?",           # Q4: Web search fallback
    "What other games has that company released?",                                     # Q5: Pronoun resolution
    "What was the first game I asked about?",                                          # Q6: Ordinal memory
    "Recommend a game similar to Hollow Knight available on Nintendo Switch (not Switch 2)",  # Q7: Recommendation
]

for i, q in enumerate(queries, 1):
    print(f"\n{'#'*70}")
    print(f"# Q{i}: {q}")
    print(f"{'#'*70}")
    report = agent.ask(q)
    print(report)


######################################################################
# Q1: Who developed FIFA 21?
######################################################################
  [retrieve_game] query='Who developed FIFA 21?' | 

3 results, best_similarity=0.7340
  [evaluate_retrieval] 

confidence=1.00, sufficient=True, web_search=False



UDAPLAY AGENT REPORT

--- Answer ---
FIFA 21 was developed by EA Vancouver, a division of Electronic Arts (EA). The game was released on October 9, 2020, and is part of the long-running FIFA series, marking its 28th installment. EA Vancouver is known for its contributions to the FIFA franchise, focusing on enhancing gameplay mechanics and introducing new features in each iteration.

Source: Internal Game Database (ChromaDB)

--- Confidence Level ---
[####################] 100%

--- Sources ---
  1. Internal Game Database (ChromaDB)

--- Tools Used ---
  retrieve_game -> evaluate_retrieval

--- Reasoning Steps ---
  Searching internal game database...
  Evaluation: confidence=1.00, sufficient=True
  Internal knowledge sufficient. Skipping web search.


######################################################################
# Q2: When was God of War Ragnarok released and on which platforms?
######################################################################


  [Query Rewrite] 'When was God of War Ragnarok released and on which platforms?' -> 'When was God of War Ragnarök released and on which platforms was God of War Ragnarök available?'
  [retrieve_game] query='When was God of War Ragnarök released and on which platforms was God of War Ragn' | 

...

  [Query Rewrite] 'Recommend a game similar to Hollow Knight available on Nintendo Switch (not Switch 2)' -> 'Recommend a game similar to Hollow Knight that is available on Nintendo Switch but not on Switch 2.'
  [retrieve_game] query='Recommend a game similar to Hollow Knight that is available on Nintendo Switch b' | 

3 results, best_similarity=0.5430
  [evaluate_retrieval] 

confidence=0.60, sufficient=False, web_search=True
  [game_web_search] query='Recommend a game similar to Hollow Knight that is available ' | 

5 results



UDAPLAY AGENT REPORT

--- Answer ---
A game similar to **Hollow Knight** that is available on **Nintendo Switch** but not on **Switch 2** is **Metroid Dread**. 

### Overview of Metroid Dread:
- **Developer**: MercurySteam and Nintendo EPD
- **Publisher**: Nintendo
- **Release Date**: October 8, 2021
- **Platforms**: Nintendo Switch
- **Genre**: Action-Adventure, Metroidvania

### Description:
**Metroid Dread** is a side-scrolling action-adventure game that continues the story of Samus Aran, the iconic bounty hunter from the Metroid series. The game features a blend of exploration, platforming, and combat, much like **Hollow Knight**. Players navigate through a mysterious alien planet, facing off against various enemies and uncovering secrets. The game emphasizes atmospheric storytelling and has a strong focus on exploration, which aligns well with the experiences found in **Hollow Knight**.

### Similarities to Hollow Knight:
- **Exploration**: Both games encourage players to explore

## 8. Agent State and Memory Inspection

In [10]:
# Inspect agent's conversation history
print("=" * 60)
print("CONVERSATION HISTORY")
print("=" * 60)
for i, msg in enumerate(agent.conversation_history):
    role = msg['role'].upper()
    content = msg['content'][:100] + "..." if len(msg['content']) > 100 else msg['content']
    print(f"\n[{i+1}] {role}: {content}")

CONVERSATION HISTORY

[1] USER: Who developed FIFA 21?

[2] ASSISTANT: FIFA 21 was developed by EA Vancouver, a division of Electronic Arts (EA). The game was released on ...

[3] USER: When was God of War Ragnarok released and on which platforms?

[4] ASSISTANT: God of War Ragnarök was released on November 9, 2022. The game is available on the following platfor...

[5] USER: What is Rockstar Games working on right now?

[6] ASSISTANT: Rockstar Games is currently developing **Grand Theft Auto VI (GTA 6)**. The game was officially reve...

[7] USER: When was Hollow Knight: Silksong released and what platforms is it on?

[8] ASSISTANT: **Hollow Knight: Silksong** was released on **September 4, 2025**. The game is available on the foll...

[9] USER: What other games has that company released?

[10] ASSISTANT: Team Cherry, the indie game development studio known for its acclaimed *Hollow Knight* series, has r...

[11] USER: What was the first game I asked about?

[12] ASSISTANT: The first 

In [11]:
# Inspect agent's long-term memory
print("=" * 60)
print("LONG-TERM MEMORY")
print("=" * 60)
if agent.memory:
    for i, entry in enumerate(agent.memory):
        print(f"\n[Memory {i+1}]")
        print(f"  Query: {entry['query']}")
        print(f"  Source: {entry['source']}")
        print(f"  Timestamp: {entry['timestamp']}")
        print(f"  Answer: {entry['answer'][:150]}...")
else:
    print("\nNo web search results have been persisted to memory yet.")
    print("(Memory is populated when web search is triggered for insufficient internal knowledge)")

LONG-TERM MEMORY

[Memory 1]
  Query: When was Hollow Knight: Silksong released and what platforms is it on?
  Source: web_search
  Timestamp: 2026-07-03T20:54:39.945516
  Answer: Hollow Knight: Silksong is set for release on September 4, 2025, available on PC, Nintendo Switch, Switch 2, PS4, PS5, Xbox One, and Xbox Series X/S....

[Memory 2]
  Query: What other games has that company released?
  Source: web_search
  Timestamp: 2026-07-03T20:54:51.946930
  Answer: Team Cherry has released Hollow Knight, its sequel Hollow Knight: Silksong, and the expansion Lifeblood. They also developed the game Interstellaria....

[Memory 3]
  Query: Recommend a game similar to Hollow Knight available on Nintendo Switch (not Switch 2)
  Source: web_search
  Timestamp: 2026-07-03T20:55:13.759318
  Answer: A game similar to Hollow Knight on Nintendo Switch is Metroid Dread. It's not available on Switch 2....


In [12]:
# Show state transition history
print("=" * 60)
print("STATE TRANSITIONS (Last Query)")
print("=" * 60)
for transition in agent.state_history:
    print(f"  {transition}")

STATE TRANSITIONS (Last Query)
  retrieve -> evaluate
  evaluate -> web_search
  web_search -> generate
  generate -> complete
